In [0]:
import requests
import json
import os
from datetime import datetime, timedelta

from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time
from urllib.parse import urlparse, parse_qs
import sys


def get_past_date(time: int) -> str:
    """ takes the time paramter as the starting time 
    e.g. 
    get_past_date(1) → "2026-03-16T01:00" (1 AM)
    """
    date = (datetime.now() - timedelta(1)).replace(hour=time, minute=0, second=0, microsecond=0)
    return date.strftime("%Y-%m-%dT%H:%M")

def directory_exist(directory: str) -> bool:
    try:
        dbutilis.fs.ls(directory)
        return True
    except Exception:
         return False      

def is_databricks_notebook()->bool:
    try:
        dbutils  # noqa: F821
        return True
    except NameError:
        return False

def update_offset(recordLimit: int, offset: int, TotalCount: int)-> str: 
        if recordLimit < TotalCount:
            offset = offset + recordLimit
            return offset
        return offset 
    
def timeout_api_restriction(responseCode: int )->bool:
        if responseCode == 503 or responseCode == 504 or responseCode == 429:
            print(f"api restriction: {responseCode}")
            time.sleep(4)
            return True
        return False
        
def versioning_fileNames(filename: str, offset:int ) -> str:
        """ differ by milliseconds , offset and unique key e.g. airports_2026-03-12-15-34-21-482_off200_a1f9c3.json"""
        unique_key = uuid.uuid4().hex[:6]
        timestamp = datetime.now().strftime("%Y-%m-%d-%H-%M-%S-%f")[:-3]
        versioned_filename = f"{filename}_{timestamp}_{offset}_{unique_key}"
        return versioned_filename
    
def loop_until_data_pool_finished(Totaldata: int, recordLimit: int)->bool:
        if Totaldata > recordLimit:
            return True
        else:
            False

def reset_timeout_rounds(timeout_rounds:int)->int:
    timeout_rounds = 0
    return timeout_rounds

def save_json_locally(
    json_data: Any,
    base_filename: str,
    offset: int,
    local_folder: Optional[str] = None,
    ) -> None:
    """
    save json localy 
    """
    versioned_filename = versioning_fileNames(base_filename, offset)

    if local_folder is not None:
        os.makedirs(local_folder, exist_ok=True)
        file_path = os.path.join(local_folder, versioned_filename)
    else:
        file_path = versioned_filename

    with open(file_path, "w", encoding="utf-8") as file:
        json.dump(json_data, file, indent=2, ensure_ascii=False)

    print(file_path)
    print(f"saved locally: {file_path}")



def save_in_Notebooks(FileName_base: str,
                      json_data: Any,
                      offset:int ,
                      directory: str 
    )-> None:
    versioned_filename = versioning_fileNames(FileName_base, offset)
    if not directory_exist(directory):
        dbutils.fs.mkdirs(directory)
    file_path = f"{directory}/{versioned_filename}"
    with open (file_path, "w") as file:
        json.dump(json_data, file, indent=2)
    print(f"{file_path} saved")




def get_next_endpoint_from_response_flightoperations(json_data: dict, meta_data_key: str) -> Optional[str]:
    """
    read api respond json file for link @Rel == 'next' and
    changes it to a Endpoint for proxy 

    config for flightoperations -> has more hrefs
    """
    counter = 0
    airport_resource = json_data.get(meta_data_key, {})
    meta = airport_resource.get("Meta", {})
    links = meta.get("Link", [])

    # if Link is a object not an array 
    if isinstance(links, dict):
        links = [links]

    for link in links:
        counter +=1
        if link.get("@Rel") == "next":
            next_href = link.get("@Href")
            if next_href.startswith("https://api.lufthansa.com"):
                return next_href.replace("https://api.lufthansa.com", "")
            
        
        elif link.get("@Rel") == "nextRange":
            next_href = link.get("@Href")
            if next_href.startswith("https://api.lufthansa.com"):
                return next_href.replace("https://api.lufthansa.com", "")
    if counter == 6:
        return "Done"
        #if next and last is missing only 4 links are available
            #"@Href": "https://api.lufthansa.com/v1/mds-references/airports?limit=100&offset=1500",
            #"@Rel": "next"
            

    return None

def get_next_endpoint_from_response(json_data: dict, meta_data_key: str) -> Optional[str]:
    """
    read api respond json file for link @Rel == 'next' and
    changes it to a Endpoint for proxy 
    """
    counter = 0
    airport_resource = json_data.get(meta_data_key, {})
    meta = airport_resource.get("Meta", {})
    links = meta.get("Link", [])

    # if Link is a object not an array 
    if isinstance(links, dict):
        links = [links]

    for link in links:
        counter +=1
        if link.get("@Rel") == "next":
            next_href = link.get("@Href")
            if next_href.startswith("https://api.lufthansa.com"):
                return next_href.replace("https://api.lufthansa.com", "")
            
        
        elif link.get("@Rel") == "last":
            last_href = link.get("@Href")
            print(f"found only last href {last_href}", file=sys.stderr)
            return None
    if counter == 4:
        return "Done"
        #if next and last is missing only 4 links are available
            #"@Href": "https://api.lufthansa.com/v1/mds-references/airports?limit=100&offset=1500",
            #"@Rel": "next"
            

    return None





def find_href(json_data: dict , meta_data_key:str) -> Optional[str]:
    
    airport_resource = json_data.get(meta_data_key, {})
    meta = airport_resource.get("Meta", {})
    links = meta.get("Link", [])

    if isinstance(links, dict):
         links = [links]

    for link in links:
        if link.get("@Rel") == "self":
              working_href = link.get("@Href")
        
        working_href.find("offset")
        if not working_href:
                print("there is no next_href")
                return None
        
def extract_offset_from_endpoint(endpoint: str) -> Optional[int]:
    parsed = urlparse(endpoint)
    qs = parse_qs(parsed.query)
    value = qs.get("offset")
    if value:
        return int(value[0])
    return None

def jump_offset(endpoint: str, skipped_values: int) -> Optional[str]:
    parsed = urlparse(endpoint)
    qs = parse_qs(parsed.query)

    limit_values = qs.get("limit")
    offset_values = qs.get("offset")

    if not limit_values or not offset_values:
        return None

    limit_value = int(limit_values[0])
    offset_value = int(offset_values[0])

    new_offset = offset_value + skipped_values

    return f"{parsed.path}?limit={limit_value}&offset={new_offset}"

def processing_Error(json_data: dict, meta_data_key: str) ->bool:
    
    #if there is not the specific meta key eg. airlineResources 
    # do a extra loop after time out 
    # if "Internal Server Error or prosssing error "
    if meta_data_key not in json_data:
        print(f"Missing meta key: {meta_data_key}", file=sys.stderr)
        save_json_locally(
                    json_data=json_data,
                    base_filename=f"{meta_data_key}error.json",
                    local_folder="errorMessages",
                    offset=0
                )
        return True
    if json_data.get(meta_data_key, {}) is None:
        return True
    #if json_data.get("ProcessingErrors", {}):
    #    return True
    return False


In [0]:

import requests
import json
import os
from datetime import datetime

from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time
from urllib.parse import urlparse, parse_qs
import sys

# from utilis import (
#     is_databricks_notebook,
#     directory_exist,
#     update_offset,
#     timeout_api_restriction,
#     versioning_fileNames,
#     loop_until_data_pool_finished,
#     reset_timeout_rounds,
#     save_json_locally,
#     save_in_Notebooks,
#     get_next_endpoint_from_response,
#     find_href,
#     extract_offset_from_endpoint,
#     jump_offset,
#     processing_Error
# )

         

def get_data_all_Reference(
    base_Url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str,

    mds_reference: str,

    recordLimit: int, 
    offset: int,

    save_on_Databricks:bool,
    meta_data_key =str,
    local_folder = str,
    ):
    
    
    dic = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{mds_reference}/"
    FileName_base = f"{mds_reference}"
    endpoint = f"/v1/mds-references/{mds_reference}?limit={recordLimit}&offset={offset}"
    
    dummy_count = 0
    timeout_rounds = 0
    proxy_error = False

    while True:
        print(f"sending request {dummy_count}")
        dummy_count += 1
        print(f"{base_Url}{endpoint}")
        try:
            response = requests.get(
                base_Url + endpoint,
                headers=headers,
                timeout=10
            )

            if response.status_code in (503, 504, 429):
                timeout_rounds = timeout_api_restriction(response.status_code)
                if timeout_rounds == 5:
                    raise Exception("infinite Loop")
                continue
            
            timeout_rounds = reset_timeout_rounds(timeout_rounds)
            if response.status_code != 200:
                raise Exception(f"new error code: {response.status_code}")
            
            print("response.url =", response.url)
            json_data = response.json()
            
            if processing_Error(json_data, meta_data_key):
                if proxy_error is True:
                    raise BrokenPipeError
                time.sleep(10)
                proxy_error = True
                continue



            if(save_on_Databricks == False):
                save_json_locally(
                    json_data=json_data,
                    base_filename=FileName_base,
                    local_folder=local_folder,
                    offset=offset
                )
            
            if(save_on_Databricks == True ):
                save_in_Notebooks(
                    mds_reference,
                    json_data,
                    offset,
                    dic)
                
            #endpoint_backup = endpoint
            offset = extract_offset_from_endpoint(endpoint)
            endpoint =get_next_endpoint_from_response(json_data, meta_data_key)
            
            if endpoint == "Done":
                return f"all files successfuly saved"
            
        except Exception as e:
            print(f"{mds_reference}: api called failed \n \
                   last api endpoint {endpoint} {e}", file=sys.stderr)
            return


In [0]:

import requests
import json
import os
from datetime import datetime

from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time
from urllib.parse import urlparse, parse_qs
import sys

# from utilis import (
#     is_databricks_notebook,
#     directory_exist,
#     update_offset,
#     timeout_api_restriction,
#     versioning_fileNames,
#     loop_until_data_pool_finished,
#     reset_timeout_rounds,
#     save_json_locally,
#     save_in_Notebooks,
#     get_next_endpoint_from_response,
#     find_href,
#     extract_offset_from_endpoint,
#     jump_offset,
#     processing_Error,
#     get_next_endpoint_from_response_flightoperations
# )

#goal: get delays of lufthansa depatures  , to see parters between Airlines 
#GET /operations/flightstatus/departures/{airportCode}/{fromDateTime}?serviceType={serviceType}

# compare FRA with MUC in delays (differ later, passanger,and so on)

def get_data_flight_depatures(
    base_Url: str,
    headers: Dict[str, str],
    catalog_name: str,
    schema_name: str,
    volume_name: str,

    operations: str,
    operation_type: str,
    operation_subtype: str,

    recordLimit: int, 
    offset: int,

    save_on_Databricks: bool,
    local_folder: str,
    
	meta_data_key:str,
	airport_code:str,
    Date=str,
	
    serviceType=str

    ):



    """ Get FRA departures only """

    dic = f"/Volumes/{catalog_name}/{schema_name}/{volume_name}/{operations}/{operation_type}/{operation_subtype}/{airport_code}/{Date}/"
    FileName_base = f"{operation_subtype}_{airport_code}"
    endpoint = f"/v1/{operations}/{operation_type}/{operation_subtype}/{airport_code}/{Date}?serviceType={serviceType}"
    dummy_count = 0
    
    timeout_rounds = 0
    proxy_error = False
    Server_error = 0

    while True:
            print(f"sending request {dummy_count}")
            dummy_count += 1
            print(f"{base_Url}{endpoint}")
            try:
                response = requests.get(
                    base_Url + endpoint,
                    headers=headers,
                    timeout=10
                )

                if response.status_code in (503, 504, 429):
                    timeout_rounds = timeout_api_restriction(response.status_code)
                    if timeout_rounds == 5:
                        raise Exception("infinite Loop")
                    continue
                
                timeout_rounds = reset_timeout_rounds(timeout_rounds)
                if response.status_code != 200:
                    raise Exception(f"new error code: {response.status_code}")
                
                print("response.url =", response.url)
                json_data = response.json()
                
                if processing_Error(json_data, meta_data_key):
                    if proxy_error is True:
                        raise BrokenPipeError
                    time.sleep(10)
                    proxy_error = True
                    continue



                if(save_on_Databricks == False):
                    save_json_locally(
                        json_data=json_data,
                        base_filename=FileName_base,
                        local_folder=local_folder,
                        offset=offset
                    )
                
                if(save_on_Databricks == True ):
                    save_in_Notebooks(
                        operation_type,
                        json_data,
                        offset,
                        dic)
                    
                #endpoint_backup = endpoint
                offset = extract_offset_from_endpoint(endpoint)
                endpoint =get_next_endpoint_from_response_flightoperations(json_data, meta_data_key)
                
                if endpoint == "Done":
                    return f"all files successfuly saved"
                
            except Exception as e:
                print(f"{operation_type}: api called failed \n \
                    last api endpoint {endpoint} {e}", file=sys.stderr)
                return

In [0]:
import requests
import json
import os
from datetime import datetime, timedelta
from pathlib import Path
from typing import Dict, Any, List, Optional
import uuid
import time
# from dotenv import load_dotenv

# from get_all_Recources import get_data_all_Reference
# import get_flight_schedules
# import get_flight_route
# from utilis import is_databricks_notebook ,get_past_date
# from get_flight_route import get_data_flight_route
# from get_flight_departures import get_data_flight_depatures

# Load .env from parent directory
# load_dotenv('../.env')

# password = dbutils.secrets.get(scope="lh-api", key="password")

def get_References()->None:
    base_url = "https://lh-proxy.onrender.com"
    if is_databricks_notebook():
        headers ={"password": dbutils.secrets.get(scope="lufthansa-api", key="LUFTHANSA_SECRET")}
    else:
        headers_env = os.getenv('headers_env')
        if not headers_env:
            raise ValueError("headers_env not found in .env file")
        headers = json.loads(headers_env)

    catalog_name ="data_catalog"
    schema_name = "bronze"
    volume_name = "bronze_volume"
        
        

    recordLimit = 100
    offset =0
    # save_on_Databricks=True
    save_on_Databricks=is_databricks_notebook()
    mds_reference = ["countries", "cities", "airports","airlines", "aircraft"]
    meta_data_key = ["CountryResource", "CityResource", "AirportResource","AirlineResource", "AircraftResource"]
    local_folder = ["Countries", "Cities", "Airports","Airlines", "Aircrafts"]

    for ref, key, folder in zip(mds_reference, meta_data_key, local_folder):
        get_data_all_Reference(
            base_Url=base_url,
            headers=headers,
                
            catalog_name=catalog_name,
            schema_name=schema_name,
            volume_name=volume_name,
            mds_reference=ref,
            recordLimit=recordLimit,
            offset=offset,

            save_on_Databricks=save_on_Databricks,
            meta_data_key=key,
            local_folder=folder
        )



def get_single_flight_route()->None:
        base_url = "https://lh-proxy.onrender.com"
        if is_databricks_notebook():
            headers ={"password": dbutils.secrets.get(scope="lufthansa-api", key="LUFTHANSA_SECRET")}
        else:
            headers_env = os.getenv('headers_env')
            if not headers_env:
                raise ValueError("headers_env not found in .env file")
        headers = json.loads(headers_env)
        catalog_name ="data_catalog"
        schema_name = "bronze"
        volume_name = "bronze_volume"
        operations="operations"
        operation_type="flightstatus"
        operation_subtype="route"
        origin="FRA"
        destination="MUC"
        #todays date
        Date="2026-03-12"
        meta_data_key="ScheduleResource"
        recordLimit=20
        offset=0
        #Date=datetime.now().strftime("%Y-%m-%d")

        serviceType="all"


        origin="FRA"
        destination ="MUC"

        get_data_flight_route(
            base_Url=base_url,
            headers=headers,
                
            catalog_name=catalog_name,
            schema_name=schema_name,
            volume_name=volume_name,

            operations= operations,
            operation_type=operation_type,

            recordLimit=recordLimit,
            offset=offset,

            save_on_Databricks=False,
            local_folder=f"flight_{operation_type}",
            
            meta_data_key=meta_data_key,
            origin=origin,
            destination=destination,
            Date=Date,
            
            serviceType=serviceType

    )
        

def get_flights()->None:
    try:
        base_url = "https://lh-proxy.onrender.com"
        if is_databricks_notebook():
                headers ={"password": dbutils.secrets.get(scope="lufthansa-api", key="LUFTHANSA_SECRET")}
        else:
            headers_env = os.getenv('headers_env')
            if not headers_env:
                raise ValueError("headers_env not found in .env file")
    except Exception as e:
        print("Error: {e}")
    catalog_name ="data_catalog"
    schema_name = "bronze"
    volume_name = "bronze_volume"
        
    operations="operations"
    operation_type="flightstatus"
    operation_subtype="departures"
    airport_code=["FRA", "MUC"]
    #todays date
    # 
    Date=get_past_date(time=0)
    meta_data_key="FlightStatusResource"
    recordLimit=20
    offset=0
    #Date=datetime.now().strftime("%Y-%m-%d")
    local_folder=f"{operation_subtype}_{airport_code}"
    serviceType="all"

    #for loop 
    for airport in airport_code:
        local_folder=f"{operation_subtype}_{airport}"
        get_data_flight_depatures(
            base_Url=base_url,
            headers=headers,
                
            catalog_name=catalog_name,
            schema_name=schema_name,
            volume_name=volume_name,

            operations= operations,
            operation_type=operation_type,
            operation_subtype= operation_subtype,
            recordLimit=recordLimit,
            offset=offset,

            save_on_Databricks=False,
            local_folder=local_folder,
            
            meta_data_key=meta_data_key,
            airport_code=airport,
            Date=Date,
            
            serviceType=serviceType

            )


if __name__ == "__main__":
    try:
        get_References()
        # get_flights()

        # get_single_flight_route()
    except Exception as e:
        print(f"error {e}")